Week Five Exercise. GET TO KNOW ME (RAG). retrieve related document from google drive and answer questions based on the knowledge extracted from google drive. 

Installation

In [ ]:
%pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client python-docx

Set up

In [1]:
import os
import re
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
from tqdm import tqdm
from openai import OpenAI
from docx import Document as DocxDocument
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import gradio as gr


In [ ]:
load_dotenv(override=True)

KNOWLEDGE_DIR = Path("person-knowledge")
STAGING_DIR = Path("person-knowledge-downloads")
DB_PATH = Path("me_vector_database")

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
RETRIEVE_K = 1
LLM_MODEL = os.getenv("GET_TO_KNOW_ME_LLM", "openai/gpt-4.1-nano")
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
STRUCTURE_MODEL = os.getenv("GET_TO_KNOW_ME_STRUCTURE_MODEL", "openai/gpt-4o-mini")
STRUCTURE_SYSTEM_PROMPT = """You are a helpful editor. Rewrite the following raw text into clear Markdown for a \"Get to Know Me\" knowledge base. Use headings, bullets, short paragraphs. Preserve all facts; do not add or invent. Output only the Markdown."""

api_key = os.getenv("OPEN_ROUTER_API_KEY")
openai = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key) if api_key else None
if api_key:
    print(f"OPEN_ROUTER_API_KEY:   {api_key[:12]}...")
else:
    print("OPEN_ROUTER_API_KEY not set — add to .env for chat and optional structuring")

Google Drive configuration

In [3]:
USE_GOOGLE_DRIVE = True
GOOGLE_DRIVE_FOLDER_ID = "root"
FORCE_REAUTH = False
STOP_AFTER_DOWNLOAD = False
DRIVE_DOWNLOAD_LIMIT = 40 
FILTER_BY_NAME = True
FILTER_KEYWORDS = ("cv", "cover letter", "profile")
DRIVE_ERROR_LOG = Path("drive_errors.log")

def _log_err(msg):
    with open(DRIVE_ERROR_LOG, "a", encoding="utf-8") as f:
        f.write(f"{datetime.now().isoformat()} {msg}\n")

ALLOWED_MIME = ["text/plain", "application/vnd.openxmlformats-officedocument.wordprocessingml.document", "application/vnd.google-apps.document"]
GOOGLE_DOCS_EXPORT = "application/vnd.openxmlformats-officedocument.wordprocessingml.document"
SCOPES_DRIVE = ["https://www.googleapis.com/auth/drive.readonly"]
STRUCTURE_CHUNK_CHARS, STRUCTURE_OVERLAP = 2000, 200
SKIP_STRUCTURE_IF_SHORTER_THAN = 2500
MAX_CHUNKS_PER_FILE = 100 


In [4]:

def _client_secret_path(root=None):
    r = Path(root) if root else Path.cwd()
    for n in ("client_secret.json", "credentials.json"):
        if (r / n).exists(): return r / n
    return r / "client_secret.json"

def build_drive_service():
    secret = _client_secret_path()
    token_path = secret.parent / "token.json"
    creds = None
    if token_path.exists():
        try: creds = Credentials.from_authorized_user_file(str(token_path), SCOPES_DRIVE)
        except Exception: token_path.unlink(missing_ok=True)
    if not creds or not creds.valid:
        if creds and creds.expired and getattr(creds, "refresh_token", None):
            try: creds.refresh(Request())
            except Exception: creds = None
        if not creds or not creds.valid:
            if not secret.exists(): raise FileNotFoundError(f"Put client_secret.json at {secret}")
            flow = InstalledAppFlow.from_client_secrets_file(str(secret), SCOPES_DRIVE)
            creds = flow.run_local_server(port=0)
        token_path.parent.mkdir(parents=True, exist_ok=True)
        token_path.write_text(creds.to_json(), encoding="utf-8")
    return build("drive", "v3", credentials=creds)

def list_drive_files(service, folder_id, limit=None):
    q = "trashed = false and (" + " or ".join(f"mimeType='{m}'" for m in ALLOWED_MIME) + ")"
    parent_id = (folder_id or "").strip() or "root"  # "root" = My Drive home (drive.google.com/drive/home)
    q += f" and '{parent_id}' in parents"
    files, token = [], None
    while True:
        n = min(100, (limit - len(files)) if limit else 100)
        if limit and n <= 0: break
        r = service.files().list(q=q, spaces="drive", fields="nextPageToken, files(id, name, mimeType, modifiedTime)", orderBy="modifiedTime desc", pageToken=token or "", pageSize=n).execute()
        files.extend(r.get("files", []))
        if limit and len(files) >= limit: return files[:limit]
        token = r.get("nextPageToken")
        if not token: break
    return files

def download_drive_file(service, meta, out_dir):
    fid, mime = meta["id"], meta.get("mimeType", "")
    ext = ".docx" if "document" in mime or mime == "application/vnd.google-apps.document" else ".txt"
    name = re.sub(r'[^\w\s\-.]', '_', Path(meta.get("name", fid)).stem)[:200] or "untitled"
    path = Path(out_dir) / f"{name}{ext}"
    path.parent.mkdir(parents=True, exist_ok=True)
    req = service.files().export_media(fileId=fid, mimeType=GOOGLE_DOCS_EXPORT) if mime == "application/vnd.google-apps.document" else service.files().get_media(fileId=fid)
    with open(path, "wb") as f:
        d = MediaIoBaseDownload(f, req)
        while not d.next_chunk()[1]: pass
    return path

def extract_text(path):
    path = Path(path)
    if path.suffix.lower() == ".txt": return path.read_text(encoding="utf-8", errors="replace")
    if path.suffix.lower() == ".docx":
        doc = DocxDocument(str(path))
        return "\n".join(p.text.strip() for p in doc.paragraphs if p.text.strip())
    raise ValueError("Only .txt and .docx supported")


def get_structure_splitter():
    return RecursiveCharacterTextSplitter(chunk_size=STRUCTURE_CHUNK_CHARS, chunk_overlap=STRUCTURE_OVERLAP, length_function=len)

def structure_chunk(chunk, model=STRUCTURE_MODEL, openai_client=None):
    key = os.getenv("OPEN_ROUTER_API_KEY")
    if not key: return chunk
    if openai_client is None:
        openai_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)
    result = openai_client.chat.completions.create(model=model, messages=[{"role": "system", "content": STRUCTURE_SYSTEM_PROMPT}, {"role": "user", "content": chunk}])
    return (result.choices[0].message.content or chunk).strip()

def build_knowledge(staging_dir, knowledge_dir, use_llm=True, filter_by_name=True, filter_keywords=("cv", "cover letter", "profile")):
    knowledge_dir = Path(knowledge_dir)
    knowledge_dir.mkdir(parents=True, exist_ok=True)
    raw = []
    paths = sorted(Path(staging_dir).glob("*.txt")) + sorted(Path(staging_dir).glob("*.docx"))
    if filter_by_name and filter_keywords:
        key = [k.lower() for k in filter_keywords]
        paths = [p for p in paths if any(k in p.name.lower() for k in key)]
        tqdm.write(f"Filtered to {len(paths)} file(s) containing: {', '.join(filter_keywords)}")
    for p in tqdm(paths, desc="Loading files"):
        try:
            t = extract_text(p)
            if t.strip(): raw.append((p.name, t.strip()))
        except Exception as e:
            msg = f"extract {p}: {e}"; _log_err(msg); tqdm.write(f"Error: {msg}")

    openai_client = None
    if use_llm and os.getenv("OPEN_ROUTER_API_KEY"):
        openai_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPEN_ROUTER_API_KEY"))
    openai_client = None
    splitter = get_structure_splitter()
    for name, text in tqdm(raw, desc="Building knowledge"):
        out = knowledge_dir / f"{Path(name).stem}.md"
        try:
            if len(text) > SKIP_STRUCTURE_IF_SHORTER_THAN:
                chunks = splitter.split_text(text)
                chunks = [c for c in chunks if c and c.strip()]
                if len(chunks) > MAX_CHUNKS_PER_FILE:
                    tqdm.write(f"  [{Path(name).stem}] {len(chunks)} chunks > {MAX_CHUNKS_PER_FILE}; writing as-is to save RAM.")
                    out.write_text(text, encoding="utf-8")
                    continue
                tqdm.write(f"  [{Path(name).stem}] {len(chunks)} chunk(s):")
                for i, c in enumerate(chunks):
                    tqdm.write(f"  --- Chunk {i+1}/{len(chunks)} ({len(c)} chars) ---")
                    tqdm.write((c[:600] + "...") if len(c) > 600 else c)
                    tqdm.write("")
                # Write chunk-by-chunk to avoid building one huge string in RAM
                with open(out, "w", encoding="utf-8") as f:
                    for i, c in enumerate(tqdm(chunks, desc=f"  write ({Path(name).stem})", leave=False)):
                        piece = structure_chunk(c, openai_client=openai_client)
                        f.write(piece)
                        if i < len(chunks) - 1:
                            f.write("\n\n--- CHUNK ---\n\n")
            else:
                out.write_text(text, encoding="utf-8")
        except Exception as e:
            msg = f"write {out}: {e}"; _log_err(msg); tqdm.write(f"Error: {msg}")
    return list(knowledge_dir.glob("*.md"))


In [ ]:

if USE_GOOGLE_DRIVE:
    STAGING_DIR.mkdir(parents=True, exist_ok=True)
    staging_has_files = list(Path(STAGING_DIR).glob("*.txt")) or list(Path(STAGING_DIR).glob("*.docx"))
    if staging_has_files:
        print(f"person-knowledge-download already has {len(staging_has_files)} file(s); skipping download.")
        if not STOP_AFTER_DOWNLOAD:
            build_knowledge(STAGING_DIR, KNOWLEDGE_DIR, use_llm=True, filter_by_name=FILTER_BY_NAME, filter_keywords=FILTER_KEYWORDS)
            print(f"Knowledge -> {KNOWLEDGE_DIR.resolve()}")
    else:
        if FORCE_REAUTH:
            _tp = _client_secret_path().parent / "token.json"
            if _tp.exists(): _tp.unlink(missing_ok=True); print("Cleared token.json — sign in again in browser.")
        svc = build_drive_service()
        files = list_drive_files(svc, GOOGLE_DRIVE_FOLDER_ID, limit=DRIVE_DOWNLOAD_LIMIT)
        errors = []
        for m in tqdm(files, desc="Downloading from Drive"):
            try:
                download_drive_file(svc, m, STAGING_DIR)
            except Exception as e:
                msg = f"download {m.get('name', m.get('id'))}: {e}"; _log_err(msg); errors.append(msg); tqdm.write(f"Error: {msg}")
        if errors: print(f"{len(errors)} error(s) -> {DRIVE_ERROR_LOG.resolve()}")
        print(f"Downloaded {len(files) - len(errors)}/{len(files)} files -> {STAGING_DIR.resolve()}")
        if STOP_AFTER_DOWNLOAD:
            print("Stopped after download (STOP_AFTER_DOWNLOAD=True). Set False to run build_knowledge.")
        elif files:
            build_knowledge(STAGING_DIR, KNOWLEDGE_DIR, use_llm=True, filter_by_name=FILTER_BY_NAME, filter_keywords=FILTER_KEYWORDS)
            print(f"Knowledge -> {KNOWLEDGE_DIR.resolve()}")
else:
    print("Set USE_GOOGLE_DRIVE=True to fetch from Drive.")

---
## 2. Load documents from person-knowledge 

In [ ]:
def load_docs_from_folder(folder_path):
    folder_path = Path(folder_path)
    folder_path.mkdir(parents=True, exist_ok=True)  # create if missing so you can add files or run 1b
    documents = []
    for ext in ["*.md", "*.txt"]:
        loader = DirectoryLoader(str(folder_path), glob=ext, loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
        documents.extend(loader.load())
    return documents

docs = load_docs_from_folder(KNOWLEDGE_DIR)
print(f"Loaded {len(docs)} document(s) from {KNOWLEDGE_DIR}")
if not docs:
    print(f"  (Folder is empty. Add .md or .txt files there.)")
    print(f"  Full path: {KNOWLEDGE_DIR.resolve()}")
for d in docs:
    print(f"  - {d.metadata.get('source', '?')} ({len(d.page_content)} chars)")

---
## 3. Chunking


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, length_function=len)
chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
if chunks:
    print("Example:", chunks[0].page_content[:300] + "...")

---
## 4. Embeddings in Chroma


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=str(DB_PATH))
print(f"Vector store at {DB_PATH} with {len(chunks)} chunks")

Retrieve + Generate

In [13]:
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVE_K})
SYSTEM_TEMPLATE = """You answer questions about the user based ONLY on the context below. If not in context, say so. Do not make up information.
Context:
{context}
"""

def get_context(question):
    docs = retriever.invoke(question)
    return "\n\n---\n\n".join(d.page_content for d in docs), docs

def answer_with_stream(question, history):
    context, retrieved_docs = get_context(question)
    if openai is None:
        return f"[OpenRouter not configured] Set OPEN_ROUTER_API_KEY in .env. Retrieved {len(retrieved_docs)} chunk(s).", {}, retrieved_docs
    messages = [{"role": "system", "content": SYSTEM_TEMPLATE.format(context=context)}, *[{"role": h["role"], "content": h["content"]} for h in history], {"role": "user", "content": question}]
    full, usage = "", {}
    for chunk in openai.chat.completions.create(model=LLM_MODEL, messages=messages, stream=True):
        if chunk.choices and chunk.choices[0].delta.content:
            full += chunk.choices[0].delta.content
        if chunk.usage:
            usage = {"prompt_tokens": chunk.usage.prompt_tokens, "completion_tokens": chunk.usage.completion_tokens}
    if not usage and full:
        r = openai.chat.completions.create(model=LLM_MODEL, messages=messages)
        full = r.choices[0].message.content or ""
        if r.usage:
            usage = {"prompt_tokens": r.usage.prompt_tokens, "completion_tokens": r.usage.completion_tokens}
    return full or "(no response)", usage, retrieved_docs

---
## 6. Gradio UI (streaming, token display)

Chat interface: each reply includes token usage at the bottom.

In [ ]:
def chat_with_tokens(message, history):
    if not message or not message.strip():
        return ""
    hist = []
    for h in history or []:
        if h[0]: hist.append({"role": "user", "content": h[0]})
        if h[1]: hist.append({"role": "assistant", "content": h[1]})
    answer, usage, _ = answer_with_stream(message.strip(), hist)
    if usage:
        answer += f"\n\n---\n*Tokens:completion={usage.get('completion_tokens', '?')}*"
    return answer

demo = gr.ChatInterface(chat_with_tokens, title="Get to Know Me — RAG", description="Ask about me.")
demo.launch()